## 1. Project Setup & Load Data

In [ ]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

for name, df in {
    "orders": orders,
    "items": items,
    "customers": customers,
    "products": products,
    "reviews": reviews
}.items():
    print(name, df.shape)
    print(df.head())
    print(df.isna().sum().sort_values(ascending=False).head())
    print("-" * 50)

## 2. Key ID / Duplicate Checks

In [3]:
print("orders duplicate order_id:", orders.duplicated("order_id").sum())
print("customers duplicate customer_id:", customers.duplicated("customer_id").sum())
print("products duplicate product_id:", products.duplicated("product_id").sum())

print("items duplicate order_id + order_item_id:", items.duplicated(["order_id", "order_item_id"]).sum())
print("reviews duplicate order_id count:", reviews.duplicated("order_id").sum())

orders duplicate order_id: 0
customers duplicate customer_id: 0
products duplicate product_id: 0
items duplicate order_id + order_item_id: 0
reviews duplicate order_id count: 551


## Convert Date Columns

In [4]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

## Aggregate Order Items to Order Level

In [6]:
item_summary = (
    items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

item_summary.head()

,order_id,item_count,total_price,total_freight,unique_products,unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1


## Aggregate Reviews to Order Level

In [7]:
review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_id", "nunique")
    )
    .reset_index()
)

review_summary.head()

,order_id,avg_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


## Build Final Order-Level Table

In [8]:
final_orders = (
    orders
    .merge(item_summary, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(review_summary, on="order_id", how="left")
)

final_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_price,total_freight,unique_products,unique_sellers,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,avg_review_score,review_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,8.72,1.0,1.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,22.76,1.0,1.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,19.22,1.0,1.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,27.20,1.0,1.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,8.72,1.0,1.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,5.0,1.0


In [9]:
print(final_orders.shape)
print(final_orders.isna().sum().sort_values(ascending=False).head(15))

(99441, 19)
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
total_price                       775
item_count                        775
unique_sellers                    775
unique_products                   775
total_freight                     775
avg_review_score                  768
review_count                      768
order_approved_at                 160
customer_id                         0
order_estimated_delivery_date       0
order_purchase_timestamp            0
customer_unique_id                  0
customer_zip_code_prefix            0
dtype: int64


## Feature Engineering for Order-Level Analysis

In [10]:
final_orders["order_month"] = final_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

final_orders["delivery_days"] = (
    final_orders["order_delivered_customer_date"] - final_orders["order_purchase_timestamp"]
).dt.days

final_orders["estimated_delivery_days"] = (
    final_orders["order_estimated_delivery_date"] - final_orders["order_purchase_timestamp"]
).dt.days

final_orders["delay_days"] = (
    final_orders["order_delivered_customer_date"] - final_orders["order_estimated_delivery_date"]
).dt.days

final_orders["is_delayed"] = final_orders["delay_days"] > 0

final_orders["order_value"] = final_orders["total_price"] + final_orders["total_freight"]

In [11]:
final_orders[
    [
        "order_id",
        "order_status",
        "order_month",
        "total_price",
        "total_freight",
        "order_value",
        "delivery_days",
        "delay_days",
        "is_delayed",
        "avg_review_score"
    ]
].head()

,order_id,order_status,order_month,total_price,total_freight,order_value,delivery_days,delay_days,is_delayed,avg_review_score
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10,29.99,8.72,38.71,8.0,-8.0,False,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07,118.70,22.76,141.46,13.0,-6.0,False,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08,159.90,19.22,179.12,9.0,-18.0,False,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11,45.00,27.20,72.20,13.0,-13.0,False,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02,19.90,8.72,28.62,2.0,-10.0,False,5.0


## Build Item / Product Category-Level Table

In [12]:
item_details = (
    items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(
        orders[
            [
                "order_id",
                "order_status",
                "order_purchase_timestamp",
                "order_delivered_customer_date",
                "order_estimated_delivery_date"
            ]
        ],
        on="order_id",
        how="left"
    )
    .merge(review_summary, on="order_id", how="left")
)

item_details["order_month"] = item_details["order_purchase_timestamp"].dt.to_period("M").astype(str)

item_details["delay_days"] = (
    item_details["order_delivered_customer_date"] - item_details["order_estimated_delivery_date"]
).dt.days

item_details["is_delayed"] = item_details["delay_days"] > 0

item_details.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,avg_review_score,review_count,order_month,delay_days,is_delayed
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,5.0,1.0,2017-09,-9.0,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,4.0,1.0,2017-04,-3.0,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,5.0,1.0,2018-01,-14.0,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,4.0,1.0,2018-08,-6.0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,5.0,1.0,2017-02,-16.0,False


## Save Cleaned Analysis Tables

In [13]:
from pathlib import Path

CLEAN_DIR = Path("../data/cleaned")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

final_orders.to_csv(CLEAN_DIR / "final_orders.csv", index=False)
item_details.to_csv(CLEAN_DIR / "item_details.csv", index=False)

In [14]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine("postgresql+psycopg2://chen@localhost:5432/postgres")

ModuleNotFoundError: No module named 'psycopg2'